In [4]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 137 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2

In [5]:
%%writefile cal.l
%{
#include "cal.tab.h"
#include <stdlib.h>
%}

%option noyywrap

DIGIT [0-9]+

%%

{DIGIT} {
    yylval = atof(yytext);
    return NUM;
}

[ \t] {
}

\n {
    return '\n';
}

. {
    return yytext[0];
}

%%

Writing cal.l


In [6]:
%%writefile cal.y
%{
#include <stdio.h>
#include <stdlib.h>

int yylex(void);
void yyerror(const char *s);
%}

%define api.value.type {double}

%token NUM

%left '+' '-'
%left '*' '/'
%right UMINUS

%%

statement:
    E '\n'
    {
        printf("Answer: %g\n", $1);
    }
    ;

E:
    E '+' E
    {
        $$ = $1 + $3;
    }
    | E '-' E
    {
        $$ = $1 - $3;
    }
    | E '*' E
    {
        $$ = $1 * $3;
    }
    | E '/' E
    {
        $$ = $1 / $3;
    }
    | NUM
    {
        $$ = $1;
    }
    ;

%%

int main(void)
{
    printf("Enter the expression: ");
    yyparse();
    return 0;
}

void yyerror(const char *s)
{
    printf("Invalid expression\n");
}

Writing cal.y


In [7]:
!bison -d cal.y

In [8]:
!flex cal.l

In [9]:
!gcc lex.yy.c cal.tab.c -o calc

In [10]:
!echo "2+2" | ./calc

Enter the expression: Answer: 4


In [11]:
!echo "10+20" | ./calc

Enter the expression: Answer: 30
